# Clasificador TF-IDF (paciente vs control sano)

In [ ]:
from sklearn.metrics import accuracy_score, classification_report
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline
import pandas as pd
import re
import os

In [ ]:
path = os.path.dirname(os.path.dirname(os.getcwd()))
print(path)

/home/pablo/Desktop/TFG/code


In [ ]:
system_prompt = "You are a patient that has gone to do an interview with a psychologist. The psychologist will ask you a series of questions and you will answer them in a natural way:\n"
user_prompt = "### Input:\n{question}\n\n### Expected Response:\n{answer}"

def apply_prompt(example):
    example["text"] = (
        system_prompt
        + user_prompt.format(question=example["question"], answer=example["answer"])
    )
    return example



## Loading dataset

In [ ]:
data_dir = path + '/../data/'
label_map = {'healthy': 'healthy', 'PT': 'patient'}
def load_split(split_name: str, label_prefix: str) -> pd.DataFrame:
    file_path = data_dir + f"ordered_{label_prefix}_{split_name}_dataset.json"
    df = pd.read_json(file_path, lines=True).fillna('')
    df = df.apply(apply_prompt, axis=1)
    df['label'] = label_map[label_prefix]
    df['script'] = "Q: " + df['question'] + "\n\nA: "+df['answer']
    return df

In [ ]:
splits = {}
for split in ['train', 'eval', 'test']:
    hc_df = load_split(split, 'healthy')
    pt_df = load_split(split, 'PT')
    df = pd.concat([hc_df, pt_df], ignore_index=True)
    splits[split] = df
    print(f"{split}: {len(hc_df)} HC + {len(pt_df)} PT = {len(df)} total")

train: 2756 HC + 7078 PT = 9834 total
eval: 326 HC + 750 PT = 1076 total
test: 239 HC + 778 PT = 1017 total


In [29]:
# Limpieza ligera del texto: lower, quitar marcas '&-' y corchetes.
def clean_text(text: str) -> str:
    text = text.lower()
    text = re.sub(r"&-", "", text)
    text = re.sub(r"\[[^\]]*\]", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

In [45]:
# GridSearchCV sobre train con más opciones de n-gramas, controlando el tamaño de la búsqueda
base_pipe = Pipeline([
    ('tfidf', TfidfVectorizer(preprocessor=clean_text)),
    ('clf', LogisticRegression(max_iter=1000, class_weight='balanced')),
])

param_grid_gs = {
    'tfidf__max_features': [30000,40000, 50000],
    'tfidf__ngram_range': [(i,j) for i in range(1,3) for j in range(i,5)],
    'tfidf__min_df': [1,2,3],
    'tfidf__max_df': [0.7,0.8,0.9],
}

X_tr, y_tr = splits['train']['text'], splits['train']['label']
gs = GridSearchCV(base_pipe, param_grid_gs, scoring='accuracy', cv=3, n_jobs=-1, verbose=1, refit=True)
gs.fit(X_tr, y_tr)


Fitting 3 folds for each of 189 candidates, totalling 567 fits


,estimator,Pipeline(step..._iter=1000))])
,param_grid,"{'tfidf__max_df': [0.7, 0.8, ...], 'tfidf__max_features': [30000, 40000, ...], 'tfidf__min_df': [1, 2, ...], 'tfidf__ngram_range': [(1, ...), (1, ...), ...]}"
,scoring,'accuracy'
,n_jobs,-1
,refit,True
,cv,3
,verbose,1
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,input,'content'


In [46]:

gs_results = (pd.DataFrame(gs.cv_results_)
              .sort_values('mean_test_score', ascending=False)
              [['mean_test_score','std_test_score','param_tfidf__max_features','param_tfidf__ngram_range','param_tfidf__min_df']])
print(gs.best_params_)
gs_results.head(10)

{'tfidf__max_df': 0.7, 'tfidf__max_features': 40000, 'tfidf__min_df': 2, 'tfidf__ngram_range': (1, 2)}


,mean_test_score,std_test_score,param_tfidf__max_features,param_tfidf__ngram_range,param_tfidf__min_df
29,0.651617,0.011927,40000,"(1, 2)",2
50,0.651617,0.011927,50000,"(1, 2)",2
176,0.651617,0.011927,50000,"(1, 2)",2
113,0.651617,0.011927,50000,"(1, 2)",2
92,0.651617,0.011927,40000,"(1, 2)",2
155,0.651617,0.011927,40000,"(1, 2)",2
8,0.651413,0.011894,30000,"(1, 2)",2
134,0.651413,0.011894,30000,"(1, 2)",2
71,0.651413,0.011894,30000,"(1, 2)",2
114,0.648973,0.013335,50000,"(1, 3)",2


In [47]:
# Evaluación en eval y reentrenamiento en train+eval con el mejor modelo
best_model = gs.best_estimator_
y_eval = splits['eval']['label']
pred_eval = best_model.predict(splits['eval']['text'])
print('[eval] accuracy (best):', accuracy_score(y_eval, pred_eval))
print(classification_report(y_eval, pred_eval, digits=3))
train_eval_df = pd.concat([splits['train'], splits['eval']], ignore_index=True)

y_test = splits['test']['label']
pred_test = best_model.predict(splits['test']['text'])
print('[test] accuracy (final):', accuracy_score(y_test, pred_test))
print(classification_report(y_test, pred_test, digits=3))


[eval] accuracy (best): 0.7016728624535316
              precision    recall  f1-score   support

     healthy      0.506     0.610     0.554       326
     patient      0.814     0.741     0.776       750

    accuracy                          0.702      1076
   macro avg      0.660     0.676     0.665      1076
weighted avg      0.721     0.702     0.709      1076

[test] accuracy (final): 0.6548672566371682
              precision    recall  f1-score   support

     healthy      0.378     0.728     0.498       239
     patient      0.883     0.632     0.737       778

    accuracy                          0.655      1017
   macro avg      0.631     0.680     0.617      1017
weighted avg      0.765     0.655     0.681      1017

